# AEF-Mosaic Zarr QA Notebook

In [ ]:
import gc
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

S3_PATH = "s3://us-west-2.opendata.source.coop/tge-labs/aef-mosaic"
N_SAMPLES = 50
CROP_SIZE = 512   # pixels per side; zarr fetches only the overlapping chunks via byte-range
YEAR = 2022
SEED = 42

In [ ]:
ds = xr.open_zarr(S3_PATH, storage_options={"anon": True}, consolidated=True)
emb = ds["embeddings"]
print(emb)
print(emb.encoding)

In [ ]:
# Confirm dimension names and compute linear transform from 4 scalar reads
X_DIM = "x"
Y_DIM = "y"

x0 = float(emb.coords[X_DIM][0])
x1 = float(emb.coords[X_DIM][-1])
nx = emb.sizes[X_DIM]
y0 = float(emb.coords[Y_DIM][0])
y1 = float(emb.coords[Y_DIM][-1])
ny = emb.sizes[Y_DIM]
x_res = (x1 - x0) / (nx - 1)
y_res = (y1 - y0) / (ny - 1)
print(f"x: {nx} px, {x0:.4f} to {x1:.4f}, res={x_res:.8f} deg/px")
print(f"y: {ny} px, {y0:.4f} to {y1:.4f}, res={y_res:.8f} deg/px")

In [ ]:
def utm_zone_bounds_wgs84(zone_number: int, hemisphere: str) -> dict:
    west = float((zone_number - 1) * 6 - 180)
    east = west + 6.0
    south, north = (0.0, 84.0) if hemisphere == "N" else (-80.0, 0.0)
    return {"west": west, "south": south, "east": east, "north": north}

ALL_UTM_ZONES = {
    f"{z}{h}": utm_zone_bounds_wgs84(z, h)
    for h in ("N", "S") for z in range(1, 61)
}
random.seed(SEED)
sampled_zone_names = random.sample(list(ALL_UTM_ZONES.keys()), min(N_STAT_ZONES, len(ALL_UTM_ZONES)))
sampled_zones = {name: ALL_UTM_ZONES[name] for name in sampled_zone_names}
print(f"Sampled {len(sampled_zones)} zones: {sampled_zone_names}")

In [ ]:
def get_crop(bounds: dict, size: int = CROP_SIZE) -> np.ndarray:
    """Fetch a size×size center crop for a UTM zone via byte-range requests.

    zarr v3 uses byte-range S3 requests to read only the logical chunks
    that overlap the window — no full shard download needed.
    Returns (F, size, size) int8.
    """
    half = size // 2
    cx = int(round(((bounds["west"] + bounds["east"]) / 2 - x0) / x_res))
    cy = int(round(((bounds["south"] + bounds["north"]) / 2 - y0) / y_res))
    cx = max(half, min(cx, nx - half))
    cy = max(half, min(cy, ny - half))
    tile = emb.isel({X_DIM: slice(cx - half, cx + half),
                     Y_DIM: slice(cy - half, cy + half)})
    if YEAR is not None and "time" in tile.dims:
        tile = tile.sel(time=YEAR, method="nearest")
    arr = np.array(tile)  # (F, size, size) int8
    if arr.ndim == 3 and arr.shape[-1] < arr.shape[0]:
        arr = arr.transpose(2, 0, 1)
    return arr


def stat_sample(crop_fyx: np.ndarray) -> dict:
    """Stats on raw int8 crop. No dequantize."""
    nodata = (crop_fyx == -128).any(axis=0)
    nan_pct = 100.0 * nodata.mean()
    valid = crop_fyx[:, ~nodata].astype(np.float32)
    if valid.size == 0:
        return {"nan_pct": 100.0, "min": None, "max": None,
                "mean": None, "std": None, "n_features": crop_fyx.shape[0]}
    return {
        "nan_pct": float(nan_pct),
        "min":     float(valid.min()),
        "max":     float(valid.max()),
        "mean":    float(valid.mean()),
        "std":     float(valid.std()),
        "n_features": int(crop_fyx.shape[0]),
    }


def pca_rgb(crop_fyx: np.ndarray) -> np.ndarray:
    """PCA false-color on raw int8 crop (-128 masked). Returns uint8 RGB."""
    f, h, w = crop_fyx.shape
    flat = crop_fyx.reshape(f, -1).T.astype(np.float32)  # (pixels, F)
    valid = ~(flat == -128).any(axis=1)
    data = flat[valid]
    if len(data) < 3:
        return np.zeros((h, w, 3), dtype=np.uint8)
    X = data - data.mean(axis=0)
    _, vecs = np.linalg.eigh(X.T @ X)   # covariance PCA, fast when F << pixels
    pcs = X @ vecs[:, ::-1][:, :3]       # (valid_pixels, 3)
    img = np.full((h * w, 3), np.nan, np.float32)
    img[valid] = pcs
    img = img.reshape(h, w, 3)
    def norm(b):
        p2, p98 = np.nanpercentile(b, [2, 98])
        return np.clip((b - p2) / (p98 - p2 + 1e-9), 0, 1)
    return (np.stack([norm(img[:, :, i]) for i in range(3)], axis=-1) * 255).astype(np.uint8)

In [ ]:
# Direct zarr access — bypasses xarray/dask caching to avoid OOM
import zarr

z_root = zarr.open_group(S3_PATH, mode="r", storage_options={"anon": True})
emb_z = z_root["embeddings"]  # shape (time, features, y, x)

# Resolve the time index once
t_arr = np.array(z_root["time"])
t_idx = int(np.argmin(np.abs(t_arr - YEAR))) if YEAR is not None else 0
print(f"Using time index {t_idx} (year={t_arr[t_idx]})")
print(f"emb_z shape: {emb_z.shape}, dtype: {emb_z.dtype}")


def get_crop(bounds: dict, size: int = CROP_SIZE) -> np.ndarray:
    """Fetch a size×size center crop via direct zarr indexing.

    Direct numpy-style indexing on the zarr array avoids the xarray/dask
    caching layer that caused OOM when fetching many small crops.
    Returns (F, size, size) int8.
    """
    half = size // 2
    cx = int(round(((bounds["west"] + bounds["east"]) / 2 - x0) / x_res))
    cy = int(round(((bounds["south"] + bounds["north"]) / 2 - y0) / y_res))
    cx = max(half, min(cx, nx - half))
    cy = max(half, min(cy, ny - half))
    arr = emb_z[t_idx, :, cy - half : cy + half, cx - half : cx + half]
    return np.asarray(arr)  # materialise; shape (F, size, size) int8



In [ ]:
results = []

for zone_name, bounds in sampled_zones.items():
    try:
        crop  = get_crop(bounds)          # (F, CROP_SIZE, CROP_SIZE) int8, ~16 MB
        stats = stat_sample(crop)
        rgb   = pca_rgb(crop)
        del crop
        gc.collect()
        results.append({"zone": zone_name, "rgb": rgb, "stats": stats, "error": None})
        mean_s = f"{stats['mean']:+.1f}" if stats['mean'] is not None else "--"
        min_s  = f"{stats['min']:.0f}"   if stats['min']  is not None else "--"
        max_s  = f"{stats['max']:.0f}"   if stats['max']  is not None else "--"
        print(f"  OK  {zone_name:>6s}  NaN={stats['nan_pct']:5.1f}%  "
              f"mean={mean_s}  min={min_s}  max={max_s}")
    except Exception as exc:
        results.append({"zone": zone_name, "rgb": None, "stats": None, "error": str(exc)})
        print(f"  ERR {zone_name:>6s}  {exc}")
        gc.collect()

n_ok = sum(1 for r in results if r["error"] is None)
print(f"\n{n_ok}/{len(results)} zones OK")


In [ ]:
rows = []
for r in results:
    row = {"zone": r["zone"]}
    if r["stats"]:
        row.update(r["stats"])
        row["status"] = "ok"
    else:
        row.update({"nan_pct": float("nan"), "min": float("nan"), "max": float("nan"),
                    "mean": float("nan"), "std": float("nan"), "n_features": float("nan"),
                    "status": f"error: {r['error']}"})
    rows.append(row)

df = pd.DataFrame(rows).sort_values("nan_pct", ascending=False).reset_index(drop=True)

def _style_row(row):
    color = "background-color: #ffcccc" if row["nan_pct"] > 80 else ""
    return [color] * len(row)

df.style.apply(_style_row, axis=1).format(
    {"nan_pct": "{:.1f}%", "min": "{:.0f}", "max": "{:.0f}", "mean": "{:.2f}", "std": "{:.2f}"},
    na_rep="--"
)

In [ ]:
ok_df = df[df["status"] == "ok"].sort_values("nan_pct", ascending=False)
fig, ax = plt.subplots(figsize=(16, 4))
colors = ["#d9534f" if v > 80 else "#5bc0de" for v in ok_df["nan_pct"]]
ax.bar(ok_df["zone"], ok_df["nan_pct"], color=colors, edgecolor="white", linewidth=0.5)
ax.axhline(80, color="red", linestyle="--", linewidth=1, label="80% NaN threshold")
ax.set_xlabel("UTM Zone")
ax.set_ylabel("NaN Coverage (%)")
ax.set_title("AEF-Mosaic QA — NaN Coverage per UTM Zone")
ax.set_ylim(0, 105)
ax.legend()
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
n_cols = 5
n_rows = (len(results) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = np.array(axes).flatten()

for ax, res in zip(axes, results):
    if res["rgb"] is not None:
        ax.imshow(res["rgb"])
        ax.set_title(f"UTM {res['zone']}\nNaN: {res['stats']['nan_pct']:.1f}%", fontsize=8)
    else:
        ax.text(0.5, 0.5, f"UTM {res['zone']}\nERROR",
                ha="center", va="center", transform=ax.transAxes, color="red")
    ax.axis("off")
for ax in axes[len(results):]:
    ax.axis("off")
plt.suptitle("AEF-Mosaic QA — 50 Random UTM Zones (PCA False-Color)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Visual QA